# Hyde Park Smell Investigation: Wind & Source Analysis

**Ward 5 Smell Log — October–November 2025**

Correlation of 39 neighbor-reported smell episodes with hourly weather data and known industrial point sources. Full narrative and findings are in the accompanying [README](README.md).

---
## 1. Setup and Data Loading

Load smell survey responses, hourly weather data (Open-Meteo API), and PurpleAir PM2.5 histories. All source files must be in the same directory as this notebook.

In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 120)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# ── Load smell reports ──
smell = pd.read_csv('../data/hyde_park_smell_reports_cleaned.csv')
print(f"Total smell reports: {len(smell)}")
print(f"Date range: {smell['Timestamp'].min()} to {smell['Timestamp'].max()}")
print(f"\nDescriptions reported:")
print(smell['What best-describes what you smell?'].value_counts().to_string())
print(f"\nIntensity distribution (1=weak, 5=strong):")
print(smell['How strong is the smell, from 1 (very weak) to 5 (very strong)?'].value_counts().sort_index().to_string())

In [ ]:
# ── Load weather data from Open-Meteo ──
# If you want to fetch fresh data, uncomment and run this:
# import urllib.request, json
# url = ("https://api.open-meteo.com/v1/forecast?latitude=41.7926&longitude=-87.6522"
#        "&hourly=temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,wind_direction_10m"
#        "&temperature_unit=fahrenheit&wind_speed_unit=mph"
#        "&timezone=America/Chicago&start_date=2025-10-01&end_date=2025-11-05")
# wx_raw = json.loads(urllib.request.urlopen(url).read())

# For reproducibility, the weather data is embedded here.
# (864 hourly observations, Oct 1 – Nov 5, 2025, Hyde Park)
import json
with open('../data/open_meteo_hyde_park.json') as f:
    wx_raw = json.load(f)

weather = pd.DataFrame({
    'time': pd.to_datetime(wx_raw['hourly']['time']),
    'temp_f': wx_raw['hourly']['temperature_2m'],
    'rh': wx_raw['hourly']['relative_humidity_2m'],
    'pressure_hpa': wx_raw['hourly']['surface_pressure'],
    'wind_mph': wx_raw['hourly']['wind_speed_10m'],
    'wind_dir': wx_raw['hourly']['wind_direction_10m'],
})
weather.set_index('time', inplace=True)
print(f"Weather data: {len(weather)} hourly observations")
print(f"Period: {weather.index[0]} to {weather.index[-1]}")
weather.head()

---
## 2. Classifying Reports: Real-Time vs. Retrospective

Separate "Just Now" reports (reliable timestamps) from retrospective submissions. Only real-time reports are matched to weather data.

In [ ]:
# ── Classify each report ──
when_col = 'Optional: When did you notice this smell? '
day_col = 'What day did you notice this smell?'

records = []
for _, row in smell.iterrows():
    ts = pd.to_datetime(row['Timestamp'])
    when = str(row.get(when_col, ''))
    day = str(row.get(day_col, ''))
    
    if 'Just Now' in when:
        category = 'just_now'
    elif 'hour ago' in when.lower():
        if day not in ('nan', '', 'NaT'):
            try:
                report_date = pd.to_datetime(day.strip().replace('/0035', '/2025'))
                category = 'retro_same_day' if report_date.date() == ts.date() else 'retro_different_day'
            except:
                category = 'retro_unknown'
        else:
            category = 'retro_no_date'
    elif when in ('nan', '', 'NaT'):
        category = 'early_form'  # Before the 'when' field existed
    else:
        category = 'other'
    
    records.append({
        'time': ts,
        'intensity': row['How strong is the smell, from 1 (very weak) to 5 (very strong)?'],
        'location': row['Where are you?'],
        'description': row['What best-describes what you smell?'],
        'category': category,
    })

reports = pd.DataFrame(records)

# Show the breakdown
print("Report classification:")
print(reports['category'].value_counts().to_string())
print(f"\nReal-time reports (Just Now + early form): {len(reports[reports['category'].isin(['just_now', 'early_form'])])}")
print(f"Retrospective same-day: {len(reports[reports['category'] == 'retro_same_day'])}")
print(f"Retrospective different-day (EXCLUDED): {len(reports[reports['category'] == 'retro_different_day'])}")

In [ ]:
# ── Filter to real-time reports only, merge with weather ──
realtime = reports[reports['category'].isin(['just_now', 'early_form'])].copy()
realtime['wx_hour'] = realtime['time'].dt.floor('h')
merged = realtime.merge(weather, left_on='wx_hour', right_index=True, how='left')

# Drop the one Feb 2026 report (outside weather data range)
merged = merged[merged['wx_hour'] < '2025-11-06']

print(f"Real-time reports with weather data: {len(merged)}")

---
## 3. Wind Direction Analysis

Wind direction describes where the wind is coming FROM. Our SE wind zone (67.5°–180°) covers the arc containing the Calumet industrial corridor and NW Indiana facilities.

In [ ]:
# ── Helper functions ──

def is_se_wind(deg):
    """Is this wind direction in the SE zone (67.5-180°)?"""
    if pd.isna(deg): return False
    return 67.5 <= deg <= 180

def wind_sector(deg):
    """Convert degrees to 16-point compass sector."""
    if pd.isna(deg): return 'Calm'
    sectors = ['N','NNE','NE','ENE','E','ESE','SE','SSE',
               'S','SSW','SW','WSW','W','WNW','NW','NNW']
    return sectors[int((deg + 11.25) / 22.5) % 16]

def angular_diff(a, b):
    """Smallest angle between two bearings."""
    d = abs(a - b) % 360
    return min(d, 360 - d)

# Tag each report
merged['se_wind'] = merged['wind_dir'].apply(is_se_wind)
merged['sector'] = merged['wind_dir'].apply(wind_sector)

In [ ]:
# ── The central result: SE wind enrichment ──

# What fraction of SMELL REPORTS occur during SE wind?
report_se_frac = merged['se_wind'].mean()

# What fraction of ALL HOURS have SE wind? (baseline)
wx_oct = weather['2025-10-01':'2025-11-05']
baseline_se_frac = wx_oct['wind_dir'].apply(is_se_wind).mean()

enrichment = report_se_frac / baseline_se_frac

print("╔══════════════════════════════════════════════════════╗")
print("║         WIND DIRECTION CORRELATION RESULT           ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  SE wind during smell reports: {merged['se_wind'].sum():2d}/{len(merged)} = {report_se_frac:.0%}        ║")
print(f"║  SE wind baseline (all hours): {baseline_se_frac:.0%}                 ║")
print(f"║  Enrichment ratio:             {enrichment:.1f}×                  ║")
print("╚══════════════════════════════════════════════════════╝")
print()
print(f"Interpretation: You are {enrichment:.1f}× more likely to smell the odor")
print("when the wind is blowing from the SE than at any random hour.")

In [ ]:
# ── Statistical significance test ──
# Is the SE wind enrichment statistically significant, or could it be chance?
# One-sided binomial test: under H₀, each report has a baseline_se_frac
# probability of occurring during SE wind (independent of smell).

from math import comb

n = len(merged)                    # total real-time reports
k = int(merged['se_wind'].sum())   # reports during SE wind
p = baseline_se_frac               # null probability (baseline rate)

# Exact binomial p-value: P(X >= k) where X ~ Binomial(n, p)
p_value = sum(comb(n, i) * p**i * (1 - p)**(n - i) for i in range(k, n + 1))

if p_value < 0.001:
    sig = f"Highly significant (p < 0.001)"
elif p_value < 0.01:
    sig = f"Significant (p < 0.01)"
elif p_value < 0.05:
    sig = f"Significant (p < 0.05)"
else:
    sig = f"Not significant (p >= 0.05)"

print("╔══════════════════════════════════════════════════════╗")
print("║         STATISTICAL SIGNIFICANCE TEST               ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Test: One-sided binomial                           ║")
print(f"║  H₀: P(SE wind | smell) = {p:.0%} (baseline rate)      ║")
print(f"║  Observed: {k}/{n} reports during SE wind ({k/n:.0%})       ║")
print(f"║  p-value: {p_value:.4f}                                ║")
print(f"║  {sig:<51s} ║")
print("╚══════════════════════════════════════════════════════╝")
print()
print(f"If wind direction didn't matter, there is only a {p_value:.2%}")
print(f"chance of seeing {k} or more SE-wind reports out of {n}.")
print("The correlation between SE wind and smell reports is not due to chance.")

In [ ]:
# ── Per-report detail table ──
print("Each real-time report matched to hourly wind:\n")
print(f"{'Date/Time':<18s} {'Wind':>5s} {'Dir':>4s} {'Spd':>5s} {'Int':>3s} {'SE?':>4s}")
print("-" * 50)
for _, r in merged.sort_values('time').iterrows():
    se = '✓ SE' if r['se_wind'] else '    '
    print(f"{r['time'].strftime('%m/%d %H:%M'):<18s} "
          f"{r['wind_dir']:5.0f}° {wind_sector(r['wind_dir']):>4s} "
          f"{r['wind_mph']:5.1f} {r['intensity']:3.0f} {se}")

### Wind Rose: Smell Reports vs. All Hours

In [ ]:
def wind_rose(ax, directions, title, color='steelblue'):
    bins = np.arange(0, 361, 22.5)
    counts, _ = np.histogram(directions.dropna(), bins=bins)
    pcts = counts / counts.sum() * 100
    theta = np.radians(bins[:-1])
    width = np.radians(22.5)
    bars = ax.bar(theta, pcts, width=width, bottom=0, color=color, 
                  alpha=0.7, edgecolor='white', linewidth=0.5)
    # Highlight SE bars (67.5-180°)
    for i, t in enumerate(bins[:-1]):
        if 67.5 <= t <= 180:
            bars[i].set_color('#d32f2f')
            bars[i].set_alpha(0.8)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_xticks(np.radians([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
    ax.set_title(title, fontsize=11, fontweight='bold', pad=20)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': 'polar'})
wind_rose(ax1, merged['wind_dir'], f"During Smell Reports\n(n={len(merged)})")
wind_rose(ax2, wx_oct['wind_dir'], f"All Hours Oct 1–Nov 5\n(n={len(wx_oct)})")
fig.suptitle("Wind Rose: Smell Reports vs. Baseline\n"
             "Red = SE sector (Calumet/Indiana source direction)",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/fig_wind_rose.png', dpi=150, bbox_inches='tight')
plt.show()

### Timeline: Hourly Wind Direction with Smell Reports Overlaid

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), height_ratios=[2, 1], sharex=True)

# --- Top panel: wind direction ---
colors_bg = ['#d32f2f' if is_se_wind(d) else '#cccccc' for d in wx_oct['wind_dir']]
ax1.scatter(wx_oct.index, wx_oct['wind_dir'], c=colors_bg, s=4, alpha=0.4, zorder=1)
ax1.axhspan(67.5, 180, alpha=0.08, color='red', zorder=0)
ax1.text(pd.Timestamp('2025-10-01 12:00'), 123, 'SE WIND ZONE (67.5°–180°)',
         color='#d32f2f', fontsize=9, alpha=0.5)

for _, r in merged.iterrows():
    ax1.scatter(r['time'], r['wind_dir'],
               s=r['intensity']**2 * 30,
               c='#d32f2f' if r['se_wind'] else '#1565c0',
               edgecolors='black', linewidth=0.8, zorder=3)

ax1.set_ylabel('Wind Direction (°)')
ax1.set_ylim(0, 360)
ax1.set_yticks([0, 45, 90, 135, 180, 225, 270, 315, 360])
ax1.set_yticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW', 'N'])
ax1.set_title('Hourly Wind Direction with Smell Reports\n'
              'Large circles = reports (size ∝ intensity). Red = SE wind.',
              fontweight='bold')
ax1.grid(True, alpha=0.15)

# --- Bottom panel: wind speed ---
ax2.fill_between(wx_oct.index, wx_oct['wind_mph'], alpha=0.3, color='steelblue')
ax2.plot(wx_oct.index, wx_oct['wind_mph'], color='steelblue', linewidth=0.5)
for _, r in merged.iterrows():
    ax2.axvline(r['time'], color='#d32f2f', alpha=0.3, linewidth=0.8)
ax2.set_ylabel('Wind Speed (mph)')
ax2.set_xlabel('Date (2025)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.savefig('../figures/fig_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

### Additional Statistics

In [ ]:
# ── Intensity comparison ──
se_int = merged.loc[merged['se_wind'], 'intensity'].mean()
other_int = merged.loc[~merged['se_wind'], 'intensity'].mean()
print(f"Mean intensity during SE wind: {se_int:.2f} (n={merged['se_wind'].sum()})")
print(f"Mean intensity other wind:     {other_int:.2f} (n={(~merged['se_wind']).sum()})")

# ── Wind speed comparison ──
print(f"\nMean wind speed during reports: {merged['wind_mph'].mean():.1f} mph")
print(f"Mean wind speed all hours:      {wx_oct['wind_mph'].mean():.1f} mph")

# ── Pressure comparison ──
print(f"\nMean pressure during reports: {merged['pressure_hpa'].mean():.1f} hPa")
print(f"Mean pressure all hours:     {wx_oct['pressure_hpa'].mean():.1f} hPa")
print(f"\nHigher pressure = stable atmosphere = pollutants trapped near ground level.")

# ── Quadrant breakdown ──
def quadrant(d):
    if pd.isna(d): return None
    if 45 <= d < 135: return 'E'
    if 135 <= d < 225: return 'S'
    if 225 <= d < 315: return 'W'
    return 'N'

print(f"\n{'Quadrant':<12s} {'Reports':>10s} {'Baseline':>10s} {'Enrichment':>12s}")
print("-" * 48)
for q in ['E', 'S', 'W', 'N']:
    r_frac = (merged['wind_dir'].apply(quadrant) == q).mean()
    b_frac = (wx_oct['wind_dir'].apply(quadrant) == q).mean()
    ratio = r_frac / b_frac if b_frac > 0 else 0
    print(f"  {q:<10s} {r_frac:10.0%} {b_frac:10.0%} {ratio:10.1f}×")

---
## 4. Point Source Cross-Reference

Match observed wind directions against the bearing from Hyde Park to known industrial facilities. If a source is at bearing X° and the wind is from X°, that source is upwind.

In [ ]:
# ── Define point sources ──
# Coordinates from EPA filings, OpenStreetMap, and Google Maps

HP_LAT, HP_LON = 41.794, -87.590  # center of smell report area

sources = [
    # Name, lat, lon, distance category, type, odor profile
    ('BP Whiting Refinery',           41.654, -87.499, 'Petroleum refinery',
     'H₂S, VOCs, flaring smoke — rotten egg / chemical / acrid'),
    ('Indiana Harbor Coke Company (IHCC)', 41.643, -87.454, 'Coke production',
     'Coal tar, PAHs, acrid chemical smoke — STRONG "burning plastic" match'),
    ('US Steel Gary Works',           41.613, -87.335, 'Steel mill (coke ovens)',
     'Coke/coal tar, SO₂, benzene — "burnt plastic" possible'),
    ('S.H. Bell Co (Ave O)',          41.659, -87.535, 'Bulk materials terminal',
     'Manganese dust, metals — less likely "burning plastic"'),
    ('American Zinc Recycling',       41.657, -87.553, 'Zinc smelter (EAF)',
     'Metallic/chemical smoke — POSSIBLE "burning plastic"'),
    ('RMG/Reserve Marine Terminal',   41.656, -87.539, 'Metals recycling',
     'Metal fumes, torch cutting — possible chemical'),
    ('Watco Transloading',            41.651, -87.556, 'Bulk cargo terminal',
     'Fugitive dust, PM — unlikely "burning plastic"'),
    ('Calumet WRP (MWRD)',            41.655, -87.576, 'Wastewater treatment',
     'H₂S, mercaptans — sewage/rotten egg, not plastic'),
    ('Clearing Industrial District',  41.780, -87.765, 'Mixed industrial (W)',
     'Variable — chemical plants, waste processing'),
    ('Stickney WRP (MWRD)',           41.800, -87.756, 'Wastewater treatment (W)',
     'H₂S, mercaptans — sewage/rotten egg'),
]

def haversine(lat1, lon1, lat2, lon2):
    R = 3959  # miles
    dlat, dlon = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def calc_bearing(lat1, lon1, lat2, lon2):
    dlon = math.radians(lon2-lon1)
    lat1r, lat2r = math.radians(lat1), math.radians(lat2)
    x = math.sin(dlon) * math.cos(lat2r)
    y = math.cos(lat1r)*math.sin(lat2r) - math.sin(lat1r)*math.cos(lat2r)*math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

# Build source table
src_data = []
for name, lat, lon, stype, odor in sources:
    dist = haversine(HP_LAT, HP_LON, lat, lon)
    brg = calc_bearing(HP_LAT, HP_LON, lat, lon)
    src_data.append({
        'name': name, 'lat': lat, 'lon': lon,
        'dist_mi': round(dist, 1), 'bearing': round(brg, 0),
        'sector': wind_sector(brg), 'type': stype, 'odor': odor
    })

src_df = pd.DataFrame(src_data).sort_values('bearing')
print(f"{'Source':<35s} {'Dist':>5s} {'Bearing':>8s} {'Type':<25s}")
print("-" * 80)
for _, s in src_df.iterrows():
    print(f"{s['name']:<35s} {s['dist_mi']:5.1f} {s['bearing']:5.0f}° {s['sector']:>3s}  {s['type']}")

### Source Map: Wind Arrows vs. Facility Locations

Polar plot with Hyde Park at center. Red triangles are industrial sources at their bearing and distance; blue arrows show wind direction during each smell report.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 11), subplot_kw={'projection': 'polar'})
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

# Plot sources
for _, s in src_df.iterrows():
    if s['dist_mi'] > 20: continue
    theta = math.radians(s['bearing'])
    ax.scatter(theta, s['dist_mi'], s=200, c='#d32f2f', marker='^',
              zorder=5, edgecolors='black', linewidth=0.5)
    ax.annotate(s['name'].split('(')[0].strip(), (theta, s['dist_mi']),
                textcoords="offset points", xytext=(8, 4), fontsize=7, ha='left')

# Plot wind arrows during smell reports
for _, r in merged.iterrows():
    if pd.isna(r['wind_dir']): continue
    theta = math.radians(r['wind_dir'])
    length = min(r['wind_mph'] * 0.5, 8)
    alpha = r['intensity'] / 5 * 0.5 + 0.3
    ax.annotate('', xy=(theta, 0.5), xytext=(theta, length),
                arrowprops=dict(arrowstyle='->', color='steelblue', alpha=alpha, lw=1.5))

ax.set_rmax(18)
ax.set_rticks([5, 10, 15])
ax.set_rlabel_position(45)
ax.set_yticklabels(['5 mi', '10 mi', '15 mi'], fontsize=8)
ax.set_xticks(np.radians([0, 45, 90, 135, 180, 225, 270, 315]))
ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])

ax.set_title('Industrial Sources (▲) vs. Wind During Smell Reports (→)\n'
             'Hyde Park at center. Arrows show where wind comes FROM.',
             fontsize=12, fontweight='bold', pad=25)
plt.savefig('../figures/fig_source_map.png', dpi=150, bbox_inches='tight')
plt.show()

### Angular Alignment Per Source

Angular mismatch between observed wind and the bearing to each source. Below 30° = good alignment.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
top_sources = [s for _, s in src_df.iterrows() if s['dist_mi'] < 15][:6]

for idx, src in enumerate(top_sources):
    ax = axes[idx // 3][idx % 3]
    diffs = [angular_diff(r['wind_dir'], src['bearing']) 
             for _, r in merged.iterrows() if not pd.isna(r['wind_dir'])]
    
    ax.hist(diffs, bins=np.arange(0, 181, 15), color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(30, color='green', ls='--', alpha=0.7, label='30° (good)')
    ax.axvline(45, color='orange', ls='--', alpha=0.7, label='45° (marginal)')
    n_good = sum(1 for d in diffs if d <= 30)
    ax.set_xlim(0, 180)
    ax.set_title(f"{src['name']}\n{src['dist_mi']} mi, bearing {src['bearing']:.0f}°\n"
                 f"{n_good}/{len(diffs)} reports within 30°", fontsize=9, fontweight='bold')
    ax.set_xlabel('Angular mismatch (°)')
    ax.set_ylabel('# Reports')
    if idx == 0: ax.legend(fontsize=7)

plt.suptitle('Wind Alignment Per Source (lower = better match)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/fig_source_alignment.png', dpi=150, bbox_inches='tight')
plt.show()

### Best Source Match Per Report

In [ ]:
print(f"{'Date/Time':<16s} {'Wind':>5s} {'Spd':>5s} {'i':>2s}  {'Best Match':<30s} {'Δ°':>3s} {'Dist':>5s} {'Travel':>6s}")
print("-" * 85)

for _, r in merged.sort_values('time').iterrows():
    wd, ws = r['wind_dir'], r['wind_mph']
    if pd.isna(wd): continue
    
    best_name, best_diff, best_dist = '', 999, 0
    for _, s in src_df.iterrows():
        d = angular_diff(wd, s['bearing'])
        # Simple scoring: angular diff + penalty for distance > 20mi
        score = d + (20 if s['dist_mi'] > 20 else 0)
        if score < (angular_diff(wd, best_diff) if best_diff != 999 else 999):
            best_name = s['name']
            best_diff = s['bearing']
            best_dist = s['dist_mi']
    
    diff = angular_diff(wd, best_diff)
    travel = best_dist / max(ws, 0.5)
    match_sym = '✓' if diff <= 30 else ('~' if diff <= 45 else ' ')
    print(f"{r['time'].strftime('%m/%d %H:%M'):<16s} "
          f"{wd:5.0f}° {ws:5.1f} {r['intensity']:2.0f}  "
          f"{best_name:<30s} {diff:3.0f}° {best_dist:5.1f} {travel:5.1f}h {match_sym}")

---
## 5. PM2.5 Plume Analysis (PurpleAir)

Test for actual plume transport using PM2.5 data from PurpleAir sensors along the SE corridor. We look for time-lagged spikes, distance gradients, and absence of signal during westerly-wind control cases.

In [ ]:
# ── Load PurpleAir data ──
pa = pd.read_csv('../data/purpleair_plume_history_all.csv')
pa['time'] = pd.to_datetime(pa['time_utc'])

pa_summary = (pa.groupby(['name', 'dist_mi', 'bearing'])
              .agg(hours=('pm25_avg', 'count'),
                   valid=('pm25_avg', lambda x: x.notna().sum()),
                   mean_pm25=('pm25_avg', 'mean'),
                   max_pm25=('pm25_avg', 'max'))
              .sort_values('dist_mi', ascending=False)
              .reset_index())

print(f"PurpleAir data: {len(pa)} hourly observations from {pa['name'].nunique()} sensors")
print(f"Period: {pa['time'].min()} to {pa['time'].max()}")
print()
print("Sensors along the plume path (source → Hyde Park):")
print()
for _, r in pa_summary.iterrows():
    print(f"  {r['name']:<42s} {r['dist_mi']:5.1f} mi  {r['bearing']:3.0f}°  "
          f"{r['valid']:.0f} hrs  mean={r['mean_pm25']:.1f}  max={r['max_pm25']:.1f} µg/m³")

In [ ]:
# ── Reference sensors for plume plots ──
# Five sensors at well-spaced distances, all on bearings 148°–155°.
# Colors: red (near source) → blue (Hyde Park)
plume_sensors = [
    ('Canalport (NLCEP)',  12.4, '#d32f2f'),   # near source
    ('Oliver (NLCEP)',      9.2, '#e65100'),   # mid-corridor
    ('Bug',                 6.9, '#f9a825'),   # SE Chicago
    ('Rooster',             5.4, '#2e7d32'),   # mid-path
    ('Purple-HP-1',         0.1, '#1565c0'),   # Hyde Park
]

### October 12: Plume propagation during the largest SE-wind episode

Six reports between 6:53–17:25 CT during SE wind at 3.4–5.3 mph. Expected plume travel time from source to Hyde Park: ~3–4 hours.

In [ ]:
# ── Oct 12 plume propagation ──
fig, ax = plt.subplots(figsize=(14, 6))

for sensor_name, dist, color in plume_sensors:
    mask = ((pa['name'] == sensor_name)
            & (pa['time'] >= '2025-10-12 04:00')
            & (pa['time'] <= '2025-10-12 20:00'))
    subset = pa[mask].sort_values('time')
    ax.plot(subset['time'], subset['pm25_avg'], '-o', color=color,
            markersize=5, linewidth=2,
            label=f"{sensor_name} ({dist} mi)", alpha=0.85)

# Mark smell report times (approximate UTC from CT+5)
for t in pd.to_datetime(['2025-10-12 11:53', '2025-10-12 12:16',
                          '2025-10-12 12:17', '2025-10-12 12:42',
                          '2025-10-12 15:21', '2025-10-12 17:25']):
    ax.axvline(t, color='black', alpha=0.15, linewidth=1, linestyle='--')
ax.axvline(pd.Timestamp('2025-10-12 11:53'), color='black', alpha=0.15,
           linewidth=1, linestyle='--', label='Smell reports')

ax.set_xlabel('Time (UTC)')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('October 12: PM2.5 Plume Propagation Along the SE Corridor\n'
             'Peak moves from source (red) to Hyde Park (blue) over ~4 hours',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlim(pd.Timestamp('2025-10-12 04:00'),
            pd.Timestamp('2025-10-12 20:00'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/fig_oct12_plume.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Oct 12 peak timing ──
print("October 12 peak PM2.5 by sensor (source → Hyde Park):")
print()
print(f"{'Sensor':<30s} {'Dist':>5s} {'Peak Hour (UTC)':>15s} "
      f"{'Peak PM2.5':>10s} {'Lag':>5s}")
print("-" * 70)

oct12 = pa[(pa['time'] >= '2025-10-12 04:00')
           & (pa['time'] <= '2025-10-12 20:00')].copy()
first_peak = None

for name, dist, _ in plume_sensors:
    s = oct12[oct12['name'] == name].dropna(subset=['pm25_avg'])
    if s.empty: continue
    peak = s.loc[s['pm25_avg'].idxmax()]
    if first_peak is None:
        first_peak = peak['time']
    lag = (peak['time'] - first_peak).total_seconds() / 3600
    print(f"{name:<30s} {dist:5.1f} {peak['time'].strftime('%H:%M'):>15s} "
          f"{peak['pm25_avg']:10.1f} {lag:4.0f}h")

print()
print("Wind speed during this episode: 3.4–5.3 mph (from Section 4)")
print(f"Expected travel time, 12 mi at ~3.5 mph: {12/3.5:.1f} hours")
print("Observed lag, Canalport → Hyde Park: 4 hours")
print("→ Consistent with plume transport at the observed wind speed.")

### October 25–26: Sustained SE-wind episode with distance gradient

Prolonged SE wind over two days. The propagation lag is less dramatic, but the distance gradient is clear.

In [ ]:
# ── Oct 25–26 sustained episode ──
fig, ax = plt.subplots(figsize=(14, 6))

for sensor_name, dist, color in plume_sensors:
    mask = ((pa['name'] == sensor_name)
            & (pa['time'] >= '2025-10-25 00:00')
            & (pa['time'] <= '2025-10-27 00:00'))
    subset = pa[mask].sort_values('time')
    ax.plot(subset['time'], subset['pm25_avg'], '-', color=color,
            linewidth=1.8,
            label=f"{sensor_name} ({dist} mi)", alpha=0.85)

# Mark smell reports (approximate UTC)
for t in pd.to_datetime(['2025-10-25 13:56', '2025-10-25 14:07',
                          '2025-10-25 14:39', '2025-10-26 08:00']):
    ax.axvline(t, color='black', alpha=0.15, linewidth=1, linestyle='--')
ax.axvline(pd.Timestamp('2025-10-25 13:56'), color='black', alpha=0.15,
           linewidth=1, linestyle='--', label='Smell reports')

ax.set_xlabel('Time (UTC)')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('October 25–26: Sustained PM2.5 Elevation During SE Wind\n'
             'Near-source sensors (red) consistently higher than Hyde Park (blue)',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
fig.autofmt_xdate(rotation=30)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/fig_oct25_plume.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Oct 26 peak-window distance gradient ──
peak_window = pa[(pa['time'] >= '2025-10-26 05:00')
                 & (pa['time'] <= '2025-10-26 09:00')].copy()

gradient = (peak_window.groupby(['name', 'dist_mi'])
            .agg(mean_pm25=('pm25_avg', 'mean'))
            .reset_index()
            .sort_values('dist_mi', ascending=False))

print("Oct 26 05:00–09:00 UTC — peak window distance gradient:")
print()
for _, r in gradient.iterrows():
    bar = '█' * int(r['mean_pm25'])
    print(f"  {r['name']:<35s} ({r['dist_mi']:5.1f} mi): "
          f"{r['mean_pm25']:5.1f} µg/m³  {bar}")

src = gradient[gradient['name'] == 'Canalport (NLCEP)']['mean_pm25'].values
hp = gradient[gradient['name'] == 'Purple-HP-1']['mean_pm25'].values
if len(src) > 0 and len(hp) > 0 and hp[0] > 0:
    print(f"\nSource/observer ratio: "
          f"{src[0]:.1f} / {hp[0]:.1f} = {src[0]/hp[0]:.1f}×")
    print("PM2.5 is ~2× higher near the industrial corridor "
          "than at Hyde Park,")
    print("consistent with plume dilution over 12 miles.")

### October 31: Westerly-wind control case

Westerly wind episode — no directional propagation expected from the SE corridor. Serves as a negative control.

In [ ]:
# ── Oct 31 control: time series + peak lag comparison ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5),
                                gridspec_kw={'width_ratios': [2, 1]})

# Left: time series
for sensor_name, dist, color in plume_sensors:
    mask = ((pa['name'] == sensor_name)
            & (pa['time'] >= '2025-10-31 06:00')
            & (pa['time'] <= '2025-10-31 22:00'))
    subset = pa[mask].sort_values('time')
    ax1.plot(subset['time'], subset['pm25_avg'], '-o', color=color,
             markersize=4, linewidth=2,
             label=f"{sensor_name} ({dist} mi)", alpha=0.85)

ax1.set_xlabel('Time (UTC)')
ax1.set_ylabel('PM2.5 (µg/m³)')
ax1.set_title('Oct 31 (W wind): All sensors peak simultaneously',
              fontsize=11, fontweight='bold')
ax1.legend(loc='upper left', fontsize=8)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax1.grid(True, alpha=0.3)

# Right: compare peak lag (Oct 12 vs Oct 31)
def get_peak_lags(start, end):
    lags = []
    ref = None
    for name, dist, _ in plume_sensors:
        s = pa[(pa['name'] == name)
               & (pa['time'] >= start)
               & (pa['time'] <= end)].dropna(subset=['pm25_avg'])
        if s.empty: continue
        peak_t = s.loc[s['pm25_avg'].idxmax(), 'time']
        if ref is None: ref = peak_t
        lags.append((dist, (peak_t - ref).total_seconds() / 3600))
    return lags

oct12_lags = get_peak_lags('2025-10-12 04:00', '2025-10-12 20:00')
oct31_lags = get_peak_lags('2025-10-31 06:00', '2025-10-31 22:00')

if oct12_lags:
    ax2.plot([d for d, _ in oct12_lags], [l for _, l in oct12_lags],
             'o-', color='#d32f2f', ms=8, lw=2, label='Oct 12 (SE wind)')
if oct31_lags:
    ax2.plot([d for d, _ in oct31_lags], [l for _, l in oct31_lags],
             's-', color='#666666', ms=8, lw=2, label='Oct 31 (W wind)')

ax2.set_xlabel('Distance from Hyde Park (mi)')
ax2.set_ylabel('Peak lag from nearest-source sensor (hours)')
ax2.set_title('Peak timing: SE plume vs. control',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.invert_xaxis()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/fig_oct31_control.png', dpi=150, bbox_inches='tight')
plt.show()

### Anomalous readings

The `Bug` sensor (6.9 mi, bearing 155°) recorded a peak of 103.8 µg/m³ — roughly 3× the next highest reading from any other sensor. We check whether this is a real event or a sensor artifact by comparing channels A and B.

In [ ]:
# ── Bug sensor anomaly check ──
bug = pa[pa['name'] == 'Bug'].copy()
bug_high = bug[bug['pm25_avg'] > 40].sort_values('pm25_avg', ascending=False)

print(f"Bug sensor readings > 40 µg/m³ ({len(bug_high)} hours):")
print()
for _, r in bug_high.iterrows():
    a = float(r['pm25_a']) if pd.notna(r['pm25_a']) and r['pm25_a'] != '' else None
    b = float(r['pm25_b']) if pd.notna(r['pm25_b']) and r['pm25_b'] != '' else None
    div = abs(a - b) if a is not None and b is not None else None
    div_s = f"{div:.1f}" if div is not None else "?"
    print(f"  {r['time_utc']}  avg={r['pm25_avg']:5.1f}  "
          f"ch_a={a}  ch_b={b}  |A-B|={div_s}")

print()
print("Large channel A/B divergence (>10 µg/m³) suggests a sensor artifact.")
print("Small divergence suggests a real reading — check nearby sensors "
      "for corroboration.")

### What the PM2.5 data adds

Three independent lines of evidence — plume propagation timing (Oct 12), distance gradient (Oct 25–26), and negative control (Oct 31) — corroborate the wind-direction findings. See README for discussion of limitations.

---
## 6. Interpretation

See README for full discussion of source attribution, the "burning plastic" descriptor, the westerly-wind outliers, and study limitations.

---
## Appendix: Data Dictionary

**`../data/hyde_park_smell_reports_cleaned.csv`** — Survey responses (emails removed)
| Column | Description |
|--------|-------------|
| Timestamp | Form submission time (Central) |
| Where are you? | Cross-street or address |
| What best-describes what you smell? | Free-text description |
| How strong is the smell... | Intensity 1–5 |
| Optional: Is there anything else... | Free-text comments |
| Optional: When did you notice this smell? | "Just Now" or "At least an hour ago" |
| What day did you notice this smell? | Date (for retrospective reports) |
| About what time did you notice this smell? | Time (for retrospective reports) |

**`../data/open_meteo_hyde_park.json`** — Hourly weather, Oct 1 – Nov 5, 2025
| Field | Description |
|-------|-------------|
| temperature_2m | Air temperature (°F) |
| relative_humidity_2m | Relative humidity (%) |
| surface_pressure | Barometric pressure (hPa) |
| wind_speed_10m | Wind speed at 10m (mph) |
| wind_direction_10m | Wind direction (°, meteorological: 0=N, 90=E, 180=S, 270=W) |

**`../data/purpleair_plume_history_all.csv`** — Hourly PM2.5 from PurpleAir sensors, Oct 1 – Nov 5, 2025
| Field | Description |
|-------|-------------|
| time_utc | Observation time (UTC) |
| sensor_index | PurpleAir sensor ID |
| name | Sensor name |
| dist_mi | Distance from Hyde Park (miles) |
| bearing | Bearing from Hyde Park (°, 0=N) |
| pm25_a | PM2.5, channel A (µg/m³) |
| pm25_b | PM2.5, channel B (µg/m³) |
| pm25_avg | PM2.5, average of A and B (µg/m³) |